# Fig. h0_clusters - layout variants with the merge snapshots (drafts)

Same data as `fig_h0_clusters.ipynb` (L = 12 didactic chain, C = Cstag, D = 1 - C).
Each variant adds the 11 snapshots of the merge sequence for both hierarchies, showing only
the edges of the minimum spanning tree (H0) or of the nearest-neighbour tree that are already
present. The newest edge of each snapshot is drawn darker, cluster sites are filled, and the
snapshot in which the cluster {4,5,6,8,9,10} is complete gets a blue frame.

Variants
- A: two rows of snapshots on top, network and two short vertical trees below.
- B: horizontal trees, each under its own row of snapshots, with merge k aligned to snapshot k.
- C: like A, but both trees horizontal and side by side.

Drafts only: the PNGs go to `drafts/`, not to `output/`. Once a variant is chosen it is moved
into `fig_h0_clusters.ipynb`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

HERE = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
FIG_DIR = HERE if (HERE / "data_raw").exists() else HERE.parent
sys.path.insert(0, str(FIG_DIR.parent / "_common"))
import paper_style as ps  # noqa: E402

DATA_DIR = FIG_DIR / "data_raw"
DRAFT_DIR = FIG_DIR / "drafts"
CLUSTER = "#0072B2"
EDGE_COLOR = "#8aa4b8"          # as in Fig. 1
SKIPPED = 6                     # 0-based index of site 7

ps.apply_style()
ps.verify_manifest(DATA_DIR)
z = np.load(DATA_DIR / "toy_chain.npz")
C, labels = z["C"], z["sdrg_labels"]
L = C.shape[0]
big = set(np.flatnonzero(labels == labels[4]).tolist())
D = np.clip(1.0 - C, 0.0, 2.0)
np.fill_diagonal(D, 0.0)
Dnn = np.full((L, L), 1.0)
np.fill_diagonal(Dnn, 0.0)
for i in range(L - 1):
    Dnn[i, i + 1] = Dnn[i + 1, i] = D[i, i + 1]

angles = np.deg2rad(90 - (18 + np.arange(L) * (360 - 36) / (L - 1)))
pos = np.c_[np.cos(angles), np.sin(angles)]


def hierarchy(Dm):
    """Single linkage with merge order as height, and the tree edge added at each merge."""
    Z = linkage(squareform(Dm, checks=False), method="single")
    members = {i: {i} for i in range(L)}
    edges = []
    for k, (a, b, _, _) in enumerate(Z):
        A, B = members[int(a)], members[int(b)]
        edges.append(min(((i, j) for i in A for j in B), key=lambda e: Dm[e]))
        members[L + k] = A | B
    Z[:, 2] = np.arange(1, len(Z) + 1)          # schematic: branch height = merge order
    complete = next(k for k in range(L - 1) if members[L + k] == big) if big in members.values() else None
    return Z, members, edges, complete


H = {"H0": hierarchy(D), "NN": hierarchy(Dnn)}

In [ ]:
# drawing helpers


def network(ax):
    for i in range(L):
        for j in range(i + 1, L):
            w = max(C[i, j], 0.0)
            if w >= 0.05:
                ax.plot(*pos[[i, j]].T, color=EDGE_COLOR, lw=0.3 + 3.2 * w ** 2, alpha=0.25 + 0.75 * w,
                        zorder=1, solid_capstyle="round")
    nodes(ax, s=150, fs=7)
    xs, ys = pos[SKIPPED]
    ax.annotate(r"strong field $\Gamma_7$", xy=(xs, ys), xytext=(xs - 0.15, ys - 0.55), fontsize=8, ha="center",
                arrowprops=dict(arrowstyle="->", lw=0.8, color="0.3"))
    ax.set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), aspect="equal")
    ax.axis("off")


def nodes(ax, s, fs):
    ax.scatter(*pos.T, s=s, color=[CLUSTER if i in big else "white" for i in range(L)],
               edgecolor="0.15", linewidth=0.6, zorder=3)
    if fs:
        for k, (x, y) in enumerate(pos):
            ax.text(x, y, str(k + 1), ha="center", va="center", fontsize=fs, zorder=4,
                    color="white" if k in big else "black")


def snapshot(ax, key, k, s=38, fs=4.6, title=True):
    """Tree edges after merge k (0-based) of hierarchy key."""
    _, _, edges, complete = H[key]
    for n, (i, j) in enumerate(edges[:k + 1]):
        inside = i in big and j in big
        color = CLUSTER if inside else "0.55"
        ax.plot(*pos[[i, j]].T, color=color, lw=2.2 if n == k else 1.2, alpha=1.0 if n == k else 0.55,
                zorder=2, solid_capstyle="round")
    nodes(ax, s=s, fs=fs)
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal")
    ax.set_xticks([]); ax.set_yticks([])
    framed = k == complete
    for side in ax.spines.values():
        side.set_visible(framed)
        side.set_color(CLUSTER); side.set_linewidth(1.4)
    if title:
        i, j = edges[k]
        ax.set_title(f"{k + 1}: {i + 1}–{j + 1}", fontsize=7, pad=2)


def color_leaves(ax, axis):
    for lbl in (ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()):
        i = int(lbl.get_text()) - 1
        lbl.set_color(CLUSTER if i in big else "0.3")
        lbl.set_fontweight("bold" if i in big else "normal")


def tree(ax, key, orientation="top", note=None):
    Z, members, _, _ = H[key]
    dendrogram(Z, ax=ax, orientation=orientation,
               link_color_func=lambda node: CLUSTER if members[node] <= big else "0.55",
               labels=[str(i + 1) for i in range(L)], leaf_font_size=8)
    color_leaves(ax, "x" if orientation in ("top", "bottom") else "y")
    for side in ax.spines.values():
        side.set_visible(False)
    if orientation in ("top", "bottom"):
        ax.set_yticks([]); ax.tick_params(axis="x", length=0)
    else:
        ax.tick_params(axis="y", length=0)
        ax.set_xticks(range(1, L)); ax.tick_params(axis="x", labelsize=7, length=2)
        ax.set_xlabel("merge order", fontsize=8)
    if note:
        ax.text(0.98, 0.95, note, transform=ax.transAxes, ha="right", va="top", fontsize=8, color="0.25")


NOTES = {"H0": "$H_0$: the cluster is one subtree", "NN": "nearest neighbours: the halves meet only through 7"}


def snapshot_row(fig, spec, key, label):
    sub = spec.subgridspec(1, L - 1, wspace=0.08)
    axs = [fig.add_subplot(sub[0, k]) for k in range(L - 1)]
    for k, ax in enumerate(axs):
        snapshot(ax, key, k)
    axs[0].text(-0.15, 0.5, label, transform=axs[0].transAxes, rotation=90, ha="right", va="center", fontsize=8)
    return axs

In [ ]:
# Variant A: snapshots on top (two rows), network and two short vertical trees below
fig = plt.figure(figsize=(13.2, 6.4))
outer = fig.add_gridspec(3, 1, height_ratios=(1.0, 1.0, 2.0), hspace=0.25)
rowA = snapshot_row(fig, outer[0], "H0", "$H_0$")
rowB = snapshot_row(fig, outer[1], "NN", "NN")
bottom = outer[2].subgridspec(1, 3, width_ratios=(0.8, 1.1, 1.1), wspace=0.15)
ax_net, ax_t0, ax_t1 = (fig.add_subplot(bottom[0, i]) for i in range(3))
network(ax_net)
tree(ax_t0, "H0", note="the cluster is\none subtree")
tree(ax_t1, "NN", note="the halves meet\nonly through site 7")
ps.panel_labels([rowA[0], rowB[0], ax_net, ax_t0, ax_t1])
ps.save_figure(fig, DRAFT_DIR, "variant_A")

In [ ]:
# Variant B: horizontal trees, merge k of the tree aligned with snapshot k above it
fig = plt.figure(figsize=(13.2, 7.2))
outer = fig.add_gridspec(1, 2, width_ratios=(0.22, 1.0), wspace=0.04)
ax_net = fig.add_subplot(outer[0, 0])
network(ax_net)
right = outer[0, 1].subgridspec(5, 1, height_ratios=(0.75, 1.5, 0.12, 0.75, 1.5), hspace=0.12)
letters = [ax_net]
right = [right[0], right[1], right[3], right[4]]
for r, key in enumerate(("H0", "NN")):
    sub = right[2 * r].subgridspec(1, L - 1, wspace=0.06)
    snaps = [fig.add_subplot(sub[0, k]) for k in range(L - 1)]
    for k, ax in enumerate(snaps):
        snapshot(ax, key, k)
    ax_t = fig.add_subplot(right[2 * r + 1])
    tree(ax_t, key, orientation="left", note=NOTES[key])
    # merge order k + 1 on x, centred under snapshot k (leaf stubs start at 0 and are clipped)
    ax_t.set_xlim(0.5, L - 0.5)
    ax_t.yaxis.tick_left()
    if key == "H0":                              # merge-order axis only once, under the last tree
        ax_t.set_xticks([]); ax_t.set_xlabel("")
    letters.append(snaps[0])                     # one letter per block (snapshots + tree)
ps.panel_labels(letters)
ps.save_figure(fig, DRAFT_DIR, "variant_B")

In [ ]:
# Variant C: snapshots on top, network and the two trees horizontal and side by side
fig = plt.figure(figsize=(13.2, 6.6))
outer = fig.add_gridspec(3, 1, height_ratios=(1.0, 1.0, 2.2), hspace=0.25)
rowA = snapshot_row(fig, outer[0], "H0", "$H_0$")
rowB = snapshot_row(fig, outer[1], "NN", "NN")
bottom = outer[2].subgridspec(1, 3, width_ratios=(0.8, 1.1, 1.1), wspace=0.25)
ax_net, ax_t0, ax_t1 = (fig.add_subplot(bottom[0, i]) for i in range(3))
network(ax_net)
tree(ax_t0, "H0", orientation="left")
tree(ax_t1, "NN", orientation="left")
for ax, note in ((ax_t0, "the cluster is\none subtree"), (ax_t1, "the halves meet\nonly through site 7")):
    ax.text(0.02, 0.98, note, transform=ax.transAxes, ha="left", va="top", fontsize=8, color="0.25")
ps.panel_labels([rowA[0], rowB[0], ax_net, ax_t0, ax_t1])
ps.save_figure(fig, DRAFT_DIR, "variant_C")

In [ ]:
# Variant D: like A, with the trees on top and the snapshots below as panels (d) and (e).
# Branches and tree edges inside the RG cluster are coloured by the subtree they belong to:
# one colour for the whole cluster in H0, one per half in the nearest-neighbour tree. The upper
# part of each tree and the edges outside the cluster stay grey, as in the original figure.
PARTS = ["#E69F00", "#009E73"]                         # yellow, green (Okabe-Ito)
GREY = "0.55"


def part_colours(key):
    """Colour of every merge node whose sites all lie in the cluster, by maximal subtree."""
    Z, members, _, _ = H[key]
    inside = [n for n in range(L, 2 * L - 1) if members[n] <= big]
    tops = [n for n in inside if not any(members[n] < members[m] for m in inside)]
    tops.sort(key=lambda n: min(members[n]))           # left half (sites 4-6) first
    colour = {}
    for c, t in zip(PARTS, tops):
        for n in inside:
            if members[n] <= members[t]:
                colour[n] = c
        for site in members[t]:
            colour[site] = c
    return colour


PART = {key: part_colours(key) for key in H}


def snapshot_parts(ax, key, k, s=38, fs=4.6):
    edges, col = H[key][2], PART[key]
    for n, (i, j) in enumerate(edges[:k + 1]):
        c = col.get(L + n, GREY)
        ax.plot(*pos[[i, j]].T, color=c, lw=2.4 if n == k else 1.3, alpha=1.0 if n == k else 0.8,
                zorder=2, solid_capstyle="round")
    nodes_c = [col.get(i, "white") for i in range(L)]
    ax.scatter(*pos.T, s=s, color=nodes_c, edgecolor="0.15", linewidth=0.5, zorder=3)
    for i, (x, y) in enumerate(pos):
        ax.text(x, y, str(i + 1), ha="center", va="center", fontsize=fs, zorder=4,
                color="white" if nodes_c[i] == PARTS[1] else "black")
    ax.set(xlim=(-1.25, 1.25), ylim=(-1.25, 1.25), aspect="equal")
    ax.set_xticks([]); ax.set_yticks([])
    framed = k == H[key][3]
    for side in ax.spines.values():
        side.set_visible(framed)
        side.set_color(PARTS[0]); side.set_linewidth(1.6)
    i, j = edges[k]
    ax.set_title(f"{k + 1}: {i + 1}–{j + 1}", fontsize=7, pad=2)


def tree_parts(ax, key, note):
    Z, members, _, _ = H[key]
    col = PART[key]
    dendrogram(Z, ax=ax, labels=[str(i + 1) for i in range(L)], leaf_font_size=9,
               link_color_func=lambda node: col.get(node, GREY))
    for lbl in ax.get_xticklabels():
        i = int(lbl.get_text()) - 1
        lbl.set_color(col.get(i, "0.3"))
        lbl.set_fontweight("bold" if i in big else "normal")
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_yticks([]); ax.tick_params(axis="x", length=0)
    ax.text(0.98, 0.95, note, transform=ax.transAxes, ha="right", va="top", fontsize=8, color="0.25")


fig = plt.figure(figsize=(13.2, 6.6))
outer = fig.add_gridspec(3, 1, height_ratios=(2.0, 1.0, 1.0), hspace=0.22)
top = outer[0].subgridspec(1, 3, width_ratios=(0.8, 1.1, 1.1), wspace=0.15)
ax_net, ax_t0, ax_t1 = (fig.add_subplot(top[0, i]) for i in range(3))
network(ax_net)
tree_parts(ax_t0, "H0", "the cluster is\none subtree")
tree_parts(ax_t1, "NN", "the halves meet\nonly through site 7")
rows = []
for r, key in ((1, "H0"), (2, "NN")):
    sub = outer[r].subgridspec(1, L - 1, wspace=0.08)
    axs = [fig.add_subplot(sub[0, k]) for k in range(L - 1)]
    for k, ax in enumerate(axs):
        snapshot_parts(ax, key, k)
    axs[0].text(-0.15, 0.5, "$H_0$" if key == "H0" else "NN", transform=axs[0].transAxes, rotation=90,
                ha="right", va="center", fontsize=8)
    rows.append(axs[0])
ps.panel_labels([ax_net, ax_t0, ax_t1] + rows)
ps.save_figure(fig, DRAFT_DIR, "variant_D")